# Phase 4 – Step 4.7: Two-Stage SAR Workflow — Integration Demo

**Purpose:** run the complete system on real cases: CSV → Risk Analyst → **human risk gate** →
Compliance Officer → **human compliance gate** → SAR document, with every step in the audit trail.

All logic lives in `src/sar_workflow.py`; this notebook only drives and displays it (master plan
decision 1). Streamlit (Stage 05) will reuse the same module.

**What you will see**
1. About 8 real cases: one per planted pattern, two clean customers and one with no activity
2. The workflow running case by case, with reviewer decisions at both gates
3. A status table for every case
4. One complete SAR document
5. The full audit trail of one case
6. Efficiency metrics: cost per stage and the **cost avoided** by the risk gate

> ⚠️ Section 2 makes **real API calls**: ~7 risk analyses plus a ReACT loop for each approved case.
> On `gpt-4o-mini` that is a few cents and about 3 minutes.

## 0. Setup

In [ ]:
# Phase 4 – Step 4.7: imports, data, both agents and the workflow, all sharing one audit log
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import textwrap

import pandas as pd

from src import config
from src.compliance_officer_agent import ComplianceOfficerAgent
from src.foundation_sar import DataLoader, ExplainabilityLogger, count_words
from src.llm_client import get_llm_client
from src.risk_analyst_agent import RiskAnalystAgent
from src.sar_workflow import CaseStatus, ConsoleReviewer, SARWorkflow, ScriptedReviewer

AUDIT_FILE = config.LOG_DIR / "demo_workflow.jsonl"      # logs/ is git-ignored
OUTPUT_DIR = config.PROJECT_ROOT / "outputs"             # SARs land in outputs/sars/ (git-ignored)
logger = ExplainabilityLogger(AUDIT_FILE)
cases = {c.customer.customer_id: c for c in DataLoader(logger).build_all_cases()}
client = get_llm_client()
workflow = SARWorkflow(
    RiskAnalystAgent(client, logger), ComplianceOfficerAgent(client, logger), logger,
    output_dir=OUTPUT_DIR,
)
print(f"{len(cases)} cases loaded | model={workflow.risk_agent.model} | audit -> {AUDIT_FILE.name}")

## 1. Choose the cases
The planted transaction-ID prefixes are the hidden answer key: used here only to *pick* cases and
to compare afterwards. The models never see them (Stage 02 masking).

In [ ]:
# Phase 4 – Step 4.7: one case per planted pattern + two clean customers + one with no activity
def hidden_label(case):
    if not case.transactions:
        return "(no activity)"
    planted = {t.transaction_id.split("_")[0] for t in case.transactions} - {"TXN"}
    return ", ".join(sorted(planted)) or "(clean)"


demo_ids = [next(cid for cid, c in cases.items() if hidden_label(c) == label)
            for label in ["STRUCT", "LARGE", "RAPID", "GEO", "FREQ"]]
demo_ids += [cid for cid, c in cases.items() if hidden_label(c) == "(clean)"][:2]
demo_ids.append(next(cid for cid, c in cases.items() if not c.transactions))
display(pd.DataFrame([{"customer": cid, "hidden_label": hidden_label(cases[cid]),
                       "transactions": len(cases[cid].transactions)} for cid in demo_ids]))

## 2. Run the workflow  ⚠️ real API calls
**Reviewers.** By default a `ScriptedReviewer` decides: it rejects `Low` risk at the risk gate
(with a reason) and approves everything else (Stage 04 decision 4). Set `INTERACTIVE = True` to make
the decisions yourself: a `ConsoleReviewer` shows each finding or narrative and asks *approve or
reject*. Either way the decision is a `HumanDecision`, and the workflow audits it.

In [ ]:
# Phase 4 – Step 4.7: push each case as far as it goes; the reviewers decide at both gates
INTERACTIVE = False

if INTERACTIVE:
    risk_reviewer = compliance_reviewer = ConsoleReviewer("notebook_user")
else:
    risk_reviewer = ScriptedReviewer("demo_risk_reviewer")
    compliance_reviewer = ScriptedReviewer("demo_compliance_officer")

for cid in demo_ids:
    [record] = workflow.process([cases[cid]], risk_reviewer, compliance_reviewer)
    print(f"{cid:<10} {hidden_label(cases[cid]):<14} -> {record.status}")

## 3. Where every case ended up

In [ ]:
# Phase 4 – Step 4.7: one row per case — findings, decisions and final status
rows = []
for cid in demo_ids:
    rec = next(r for r in workflow.records.values() if r.case.customer.customer_id == cid)
    risk, out = rec.risk_output, rec.compliance_output
    rows.append({
        "customer": cid,
        "hidden_label": hidden_label(rec.case),
        "classification": risk.classification if risk else None,
        "risk_level": risk.risk_level if risk else None,
        "risk_gate": rec.risk_decision.decision if rec.risk_decision else None,
        "narrative_words": count_words(out.narrative) if out else None,
        "compliance_gate": rec.compliance_decision.decision if rec.compliance_decision else None,
        "status": str(rec.status),
        "sar_id": rec.sar_id,
    })
status_table = pd.DataFrame(rows)
display(status_table)

**How to read this table:** rejected cases stop at the risk gate with no narrative and no Stage 2
cost. `MANUAL_REVIEW` would mean an agent failed and a person must take over. The zero-activity
case is answered by a rule, with no model call at all.

## 4. One complete SAR document
This is the regulatory deliverable, saved as JSON in `outputs/sars/`. It contains the subject's
**full personal details** (Stage 04 decision 2), which is why `outputs/` is git-ignored and the audit
log stays redacted. Clear this cell's output before sharing the notebook.

In [ ]:
# Phase 4 – Step 4.7: the first filed SAR, section by section
filed = [r for r in workflow.records.values() if r.status == CaseStatus.SAR_FILED]
if filed:
    sar = json.loads(Path(filed[0].sar_path).read_text(encoding="utf-8"))
    print(f"{sar['sar_id']}  ({Path(filed[0].sar_path).relative_to(ROOT)})\n")
    print("Subject:  ", sar["subject"]["name"], "|", sar["subject"]["occupation"])
    print("Accounts: ", ", ".join(f"{a['account_type']} ({a['status']})" for a in sar["accounts"]))
    a = sar["activity"]
    print(f"Activity:  {a['typology']} | {a['risk_level']} | {a['start_date']} to {a['end_date']} | "
          f"${a['total_amount']:,.2f} over {len(a['transaction_ids'])} transactions")
    print("Narrative:")
    print(textwrap.fill(sar["narrative"]["narrative"], width=100, initial_indent="  ",
                        subsequent_indent="  "))
    print("Citations:", "; ".join(sar["narrative"]["regulatory_citations"]))
    for gate in ("risk_review", "compliance_review"):
        r = sar[gate]
        print(f"{gate}: {r['decision']} by {r['reviewer_id']} at {r['decided_at'][:19]}")
    print("Provenance:", sar["provenance"])
else:
    print("No SAR filed in this run.")

## 5. The audit trail of one case

In [ ]:
# Phase 4 – Step 4.7: every event for the filed case — agents, humans and the workflow itself
events = [json.loads(line) for line in AUDIT_FILE.read_text(encoding="utf-8").splitlines()]
if filed:
    case_events = [e for e in events if e["case_id"] == filed[0].case_id]
    display(pd.DataFrame([{
        "time": e["timestamp"][11:19],
        "actor": e["agent_type"],
        "action": e["action"],
        "success": e["success"],
        "reviewer": e.get("reviewer_id"),
        "tokens": (e.get("token_usage") or {}).get("total_tokens"),
        "reasoning": textwrap.shorten(e["reasoning"] or "", 70),
    } for e in case_events]))

**This is what an examiner reads:** who (agent or person) did what, when, with which model and
prompt version, at what cost, and why. Rejections are recorded the same way as approvals.

## 6. Efficiency metrics
Token counts are exact; **prices are assumptions** for `gpt-4o-mini` (`src/metrics.py`). Check the
current price list before quoting dollar figures.

In [ ]:
# Phase 4 – Step 4.7: cost per stage, gate approval rates and the cost avoided by the risk gate
metrics = workflow.metrics()
display(pd.DataFrame(metrics.stage_rows()))
for key, value in metrics.summary().items():
    print(f"{key:<30} {value}")

In [ ]:
# Phase 4 – Step 4.7: rough projection to all 150 cases, using THIS run's rates (assumption!)
s = metrics.summary()
active = sum(1 for c in cases.values() if c.transactions)
risk_avg = metrics.risk.avg_cost_usd or 0.0
comp_avg = metrics.compliance.avg_cost_usd
rate = metrics.risk_gate.approval_rate
if comp_avg is not None and rate is not None:
    two_stage = active * (risk_avg + rate * comp_avg)
    single_stage = active * (risk_avg + comp_avg)
    print(f"{active} active cases | approval rate {rate:.0%} at the risk gate")
    print(f"two-stage   ≈ ${two_stage:.3f}")
    print(f"narrate all ≈ ${single_stage:.3f}  -> saves ≈ ${single_stage - two_stage:.3f} "
          f"({1 - two_stage / single_stage:.0%})")

## 7. Takeaways
- **The gate is enforced in code, not by convention:** the Compliance Officer cannot run on a case no
  human approved (`WorkflowError`), and no SAR can exist without two approvals (`SARDocument`).
- **Every decision is evidence:** agent runs, human approvals and rejections, manual-review routing
  and SAR generation all land in one ordered JSONL audit trail.
- **The two-stage design pays off in proportion to the rejection rate:** each case stopped at the risk
  gate saves a whole ReACT loop, the most expensive step.
- **Humans still matter:** the narratives pass every rule, but the compliance gate is where a person
  judges whether the story is fair and complete.